**NỘI DUNG THỰC HÀNH:**

I) LUCENE

1.   Cài đặt Pylucene
2.   Lập chỉ mục với Pylucene
2.   Truy xuất chỉ mục với Pylucene

II) CRAWLER


**I) LUCENE**

1. Cài đặt PyLucene:

- Download các file easy-install.pth, JCC-3.15-py3.12-linux-x86_64.egg và lucene-10.0.0-cp312-cp312-linux_x86_64.whl vào thư mục /content.


In [1]:
%%capture
%cd /content/
!curl http://www.simplyans.com/file/lucene-10.0.0-cp312-cp312-linux_x86_64.whl -o lucene-10.0.0-cp312-cp312-linux_x86_64.whl
!curl http://www.simplyans.com/file/JCC-3.15-py3.12-linux-x86_64.egg -o JCC-3.15-py3.12-linux-x86_64.egg
!curl http://www.simplyans.com/file/easy-install.pth -o easy-install.pth



- Cài đặt Openjdk-21

- Cập nhật các biến môi trường Linux.

In [2]:
%%capture
!apt update
!apt install openjdk-21-jdk
!mv /usr/lib/jvm/java-21-openjdk-amd64 /usr/lib/jvm/temurin-21-jdk-amd64

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,153 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,008 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntug

- Cài đặt các file vừa download. Sau đó, restart session để các file vừa cài đặt có tác dụng. Nếu không restart session, các package của lucene sẽ không thực thi được.

In [3]:
%%capture
%cd /content/
!cp easy-install.pth /usr/local/lib/python3.12/dist-packages/
!yes | unzip JCC-3.15-py3.12-linux-x86_64.egg -d /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/
!python -m pip install lucene-10.0.0-cp312-cp312-linux_x86_64.whl


/content
Archive:  JCC-3.15-py3.12-linux-x86_64.egg
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/libjcc3.so  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/EGG-INFO/PKG-INFO  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/EGG-INFO/SOURCES.txt  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/EGG-INFO/dependency_links.txt  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/EGG-INFO/native_libs.txt  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/EGG-INFO/not-zip-safe  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/EGG-INFO/top_level.txt  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86_64.egg/jcc/__init__.py  
  inflating: /usr/local/lib/python3.12/dist-packages/JCC-3.15-py3.12-linux-x86

2) Lập chỉ mục với Pylucene (Xem slide IR-TH3.pdf)

- Khai báo sử dụng các package liên quan đến việc lập chỉ mục.

- Lưu ý, cần phải gọi lucene.initVM() một lần duy nhất để khởi tạo máy ảo Java.

In [1]:
import jcc

In [2]:
import lucene
import os
from org.apache.lucene.store import FSDirectory
from org.apache.lucene.index import IndexWriterConfig, IndexWriter
from org.apache.lucene.analysis.standard import StandardAnalyzer
from org.apache.lucene.document import Document, TextField, StoredField, Field
from java.nio.file import Paths

lucene.initVM()

* Tạo thư mục lưu trữ chỉ mục

* lập chỉ mục cho từng tài liệu

In [3]:
if not os.path.exists("/content/idx"):
  os.makedirs("/content/idx")
directory = FSDirectory.open(Paths.get("/content/idx"))
analyzer = StandardAnalyzer()
cf = IndexWriterConfig(analyzer)
cf.setOpenMode(IndexWriterConfig.OpenMode.CREATE)
writer = IndexWriter(directory, cf)
D = [
  {
      "Title": "Modern Computer Architecture",
      "Author": "James Harrison",
      "Content": "computer architecture, modern RAM, CPU speed, hard drive capacity, easy to use."
  },
  {
      "Title": "Fashion in Use",
      "Author": "James Smith",
      "Content": "white shirt, hard hat, modern hair styles, easy to work"
  },
  {
      "Title": "Safety in Transportation",
      "Author": "Smith Johnson",
      "Content": "drunk driver, high speed vehicle architectures, carelessly drive, modern designed cars, smith"
  }
     ]

for ind in range(len(D)):
  doc = Document()
  id = StoredField("id", ind)
  title = TextField("title", D[ind]["Title"], Field.Store.YES)
  author = TextField("author", D[ind]["Author"], Field.Store.YES)
  content = TextField("content", D[ind]["Content"], Field.Store.YES)
  doc.add(id)
  doc.add(title)
  doc.add(author)
  doc.add(content)
  writer.addDocument(doc)

writer.close()

3) Truy xuất chỉ mục với Pylucene (Xem slide IR-TH3.pdf)

* Khai báo sử dụng các package liên quan đến việc truy xuất chỉ mục.

In [4]:
from org.apache.lucene.store import FSDirectory
from org.apache.lucene.index import DirectoryReader, IndexReader
from org.apache.lucene.search import IndexSearcher, TopDocs, ScoreDoc
from org.apache.lucene.analysis.standard import StandardAnalyzer
from org.apache.lucene.document import Document, TextField, StoredField, Field
from org.apache.lucene.queryparser.classic import QueryParser
from java.nio.file import Paths

* Đọc chỉ mục đã được tạo

* Tạo câu truy vấn

* Hiển thị kết quả truy xuất

In [5]:
directory = FSDirectory.open(Paths.get("/content/idx"))
analyzer = StandardAnalyzer()
reader = DirectoryReader.open(directory)
searcher = IndexSearcher(reader)
parser = QueryParser("content", analyzer)

docset = searcher.storedFields()

Q = ["architecture", "driver", "carelessly drive", "modern car"]

for q in Q:
  print("Search query: ", q)
  query = parser.parse(q)
  docs = searcher.search(query, reader.maxDoc())
  for score in docs.scoreDocs:
    doc = docset.document(score.doc)
    print("Document: ", doc["title"], "- similarity score: ", score.score)
  print("===================================")

Search query:  architecture
Document:  Modern Computer Architecture - similarity score:  0.43535494804382324
Search query:  driver
Document:  Safety in Transportation - similarity score:  0.43535494804382324
Search query:  carelessly drive
Document:  Safety in Transportation - similarity score:  0.6439727544784546
Document:  Modern Computer Architecture - similarity score:  0.20861777663230896
Search query:  modern car
Document:  Fashion in Use - similarity score:  0.06376498192548752
Document:  Modern Computer Architecture - similarity score:  0.059269800782203674
Document:  Safety in Transportation - similarity score:  0.059269800782203674


**BÀI TẬP**
1) Hãy lập chỉ mục cho tập tài liệu Cranfield với Lucene.
2) Thử nghiệm tập truy vấn của Cranfield với Lucene, đánh giá theo độ đo MAP nội suy và so sánh với kết quả trong các buổi thực hành trước đó.

**II) CRAWLER**

* Dùng package requests để tải trang HTML

* Dùng BeautifulSoup để phân tách trang HTML

* Dùng hàm find_all để tìm tất cả hyperlink trong trang web, để tiếp tục dò tìm.

* Dùng hàm find_all để tìm nội dung chính của trang.

In [6]:
from bs4 import BeautifulSoup
import requests
import re
from time import sleep

def crawl():
	LINKS = {"https://vnexpress.net/kinh-doanh": 0}
	keys = list(LINKS.keys());
	i = 0
	w = open('vnexpress-news.txt', 'wt', encoding='utf-8')
	MAX = 5 # Download 5 trang đầu tiên để lấy nội dung văn bản.
	while i < len(keys) and MAX > 0:
		if LINKS.get(keys[i]) > 0:
			continue
		print("fetch", keys[i])
		r = requests.get(keys[i])
		i += 1
		sleep(2)
		content = r.text
		page = BeautifulSoup(content, 'html.parser')
		atags = page.find_all('a')
		hrefs = []
		for a in atags:
			url = a.attrs.get('href')
			if url == None:
				continue
			if url[-5:] == '.html':
				if not url in hrefs:
					hrefs.append(url)
		for href in hrefs:
			if LINKS.get(href) == None:
				LINKS[href] = 0
				keys.append(href)

		maincontent = page.find_all('article', {'class':"fck_detail"})
		if maincontent != None:
			if len(maincontent) > 0:
				maintext = ""
				for p in maincontent:
					txts = p.getText().strip().split("\n")
					for txt in txts:
						if len(txt) > 1:
							maintext += txt + "\n"

				w.write("{}\n{}\n============================================\n".format(keys[i], maintext))
				#print(maintext)
		MAX -= 1
	w.close()

crawl()


fetch https://vnexpress.net/kinh-doanh
fetch https://vnexpress.net/ong-trump-noi-co-the-bo-thue-thu-nhap-trong-vai-nam-toi-4987248.html
fetch https://vnexpress.net/ca-voi-gom-bitcoin-sau-3-thang-ban-rong-4987203.html
fetch https://vnexpress.net/cong-ty-van-hanh-metro-ben-thanh-suoi-tien-lo-140-ty-dong-4986581.html
fetch https://vnexpress.net/ly-do-cac-hang-xe-dien-duoc-mien-tru-trach-nhiem-thu-gom-tai-che-pin-4987121.html
